# 1. Context

This Notebook evaluates Character Error Rate (CER) & Word Error Rate (WER) in OCRed Document at paragraph compared to Ground Truth

> Accuracy evaluation will be scoped to document (1 page) and multi column layput in later studies

# 2. Imports

In [1]:
import jiwer
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1
import os
import json

In [3]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

In [4]:
from pathlib import Path

In [5]:
PROJECT_ID = os.getenv("PROJECT_ID", "")
API_LOCATION = os.getenv("API_LOCATION", "")

In [7]:
indic_parser_name = "indic_test_processor"

In [8]:
opts = ClientOptions(api_endpoint=f"{API_LOCATION}-documentai.googleapis.com")
client = documentai_v1.DocumentProcessorServiceClient(client_options=opts)
full_processor_name = client.processor_path(PROJECT_ID, API_LOCATION, "5945bfe7932ca5b7")
request = documentai_v1.GetProcessorRequest(name=full_processor_name)
processor = client.get_processor(request=request)

# Language: Hindi

In [9]:
class DocumentAIOCR:
    """A class to handle Google Document AI OCR operations"""
    
    def __init__(self, parser_nm: str, processor_id: str = "5945bfe7932ca5b7"):
        """
        Initialize Google Document AI Processor
        
        Args:
            parser_nm: Name of the parser
            processor_id: ID of the Document AI processor
        """
        self.api_location = os.getenv("API_LOCATION", "")
        self.project_id = os.getenv("PROJECT_ID", "")
        
        if not self.api_location or not self.project_id:
            raise ValueError("API_LOCATION and PROJECT_ID environment variables must be set")
            
        self.client = self._initialize_client()
        self.processor = self._initialize_processor(processor_id)
    
    def _initialize_client(self) -> documentai_v1.DocumentProcessorServiceClient:
        """Initialize and return Document AI client"""
        client_options = ClientOptions(
            api_endpoint=f"{self.api_location}-documentai.googleapis.com"
        )
        return documentai_v1.DocumentProcessorServiceClient(
            client_options=client_options
        )
    
    def _initialize_processor(self, processor_id: str) -> documentai_v1.Processor:
        """Initialize and return Document AI processor"""
        processor_name = self.client.processor_path(
            self.project_id, 
            self.api_location, 
            processor_id
        )
        request = documentai_v1.GetProcessorRequest(name=processor_name)
        return self.client.get_processor(request=request)
    
    def _read_image(self, img_path: str) -> bytes:
        """Read image file and return bytes content"""
        try:
            with open(img_path, "rb") as image:
                return image.read()
        except IOError as e:
            raise IOError(f"Error reading image file {img_path}: {str(e)}")
    
    def perform_ocr(self, img_path: str) -> documentai_v1.Document:
        """
        Perform OCR for a given image path
        
        Args:
            img_path: Path to the image file
            
        Returns:
            Document object containing OCR results
        """
        img_content = self._read_image(img_path)
        raw_document = documentai_v1.RawDocument(
            content=img_content,
            mime_type="image/png"
        )
        
        try:
            request = documentai_v1.ProcessRequest(
                name=self.processor.name,
                raw_document=raw_document
            )
            result = self.client.process_document(request=request)
            return result.document
        except Exception as e:
            raise RuntimeError(f"OCR processing failed: {str(e)}")
    
    async def perform_ocr_async(self, img_path: str) -> documentai_v1.Document:
        """Asynchronous version of perform_ocr"""
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.perform_ocr, img_path)
    
    async def process_multiple_images(self, img_paths: list) -> list:
        """
        Process multiple images concurrently
        
        Args:
            img_paths: List of image paths to process
            
        Returns:
            List of Document objects containing OCR results
        """
        tasks = [self.perform_ocr_async(img_path) for img_path in img_paths]
        return await asyncio.gather(*tasks, return_exceptions=True)

In [10]:
# Create DocumentAIOCR instance
document_ai_obj = DocumentAIOCR(parser_nm=indic_parser_name)

In [124]:
_PATH_SYNTH_DATA_ = Path("../../synth_data")
_LANG_ = "bengali"
_PATH_IMAGES_LANG_ = _PATH_SYNTH_DATA_.joinpath(_LANG_)

_PATH_IMAGES_LIST_PNG_ = list(_PATH_IMAGES_LANG_.glob("images/*/*.png"))
_PATH_JSON_LIST_GT_ = list(_PATH_IMAGES_LANG_.glob("gt/*.json"))

## 2. Handling MultiPage Documents

## 2.1. Get Single page PDF Images

In [120]:
def get_single_page_doc_name(path_pdf_images: list[Path]) -> list[str]:
    """Get names of document which have multiple pages. This helps in handling these files in downstream"""    
    # dictionary to store, file_name and it counts as they appear in image files. count >1 indicate, pdf contains 2 files
    file_name_img_counts = {} 
    path_pdf_img_single_pg = []
    for image_fp in path_pdf_images:
        image_fn = image_fp.name.split(".")[0]
        file_name, pdf_page_count = image_fn.split("_n_pages_")[0], int(image_fn.split("_n_pages_")[-1].split("_")[0])
        file_name_img_counts[file_name] = pdf_page_count
        if pdf_page_count == 1:
            path_pdf_img_single_pg.append(image_fp)

    return file_name_img_counts , path_pdf_img_single_pg

## 2.2. Get GT Json corresponding to single page jsons

In [ ]:
# get gt json corresponding to single page pdfs
def get_single_page_gt_jsons(path_gt_jsons: list[Path], fn_page_count: dict[str, int]) -> list[Path]:
    """Gets GT Json file path for documents contained in 1 page"""
    
    fn_counts_single_page = [key for key, value in fn_page_count.items() if value == 1]
    path_gt_single_pg = []
    for gt_json_path in path_gt_jsons:
        gt_file_name = gt_json_path.name.split(".")[0]
        if gt_file_name in fn_counts_single_page:
            path_gt_single_pg.append(gt_json_path)
    return path_gt_single_pg

In [126]:
fn_counts, path_images_single_pg = get_single_page_doc_name(_PATH_IMAGES_LIST_PNG_)
path_gt_single_pg = get_single_page_gt_jsons(_PATH_JSON_LIST_GT_, fn_counts)

In [127]:
# Process multiple images concurrently
results = asyncio.run(document_ai_obj.process_multiple_images(path_images_single_pg))

In [129]:
ocr_results_list_dict = []
for IDX, IMG_PATH in enumerate(path_images_single_pg):
    degradation_level = IMG_PATH.parent.name
    gt_file_name = IMG_PATH.name.split("_n_pages_")[0]
    ocr_results_list_dict.append({"file_id": gt_file_name, "degradation_level": degradation_level.split("_")[0][0] + "_" + degradation_level.split("_")[1] , "ocr_output_raw": results[IDX].text})

In [131]:
import pandas as pd

In [132]:
df = pd.DataFrame(ocr_results_list_dict)

In [133]:
def get_ocr_results_df(image_paths, results):
    ocr_results_list_dict = []
    for IDX, IMG_PATH in enumerate(image_paths):
        degradation_level = IMG_PATH.parent.name
        gt_file_name = IMG_PATH.name.split("_n_pages_")[0]
        ocr_results_list_dict.append({"file_id": gt_file_name, "degradation_level": degradation_level.split("_")[0][0].upper() + "_" +degradation_level.split("_")[1] , "ocr_output_raw": results[IDX].text})

    df_result = pd.DataFrame(ocr_results_list_dict)

    # pivot#
    pivoted_df = df_result.pivot(index='file_id',columns='degradation_level', values='ocr_output_raw').reset_index()

    # Rename columns to include 'ocr_output_' prefix
    pivoted_df.columns.name = None  # Remove the columns name
    renamed_columns = {
        col: f'ocr_output_{col}' if col != 'file_id' else col 
        for col in pivoted_df.columns
    }
    pivoted_df = pivoted_df.rename(columns=renamed_columns)

    return pivoted_df

In [136]:
ocr_res_df = get_ocr_results_df(path_images_single_pg, results)

In [138]:
def read_json(path) -> dict:
    """
    Reads JSON and properly decodes Indic text
    
    Args:
        path: Path to JSON file
    Returns:
        dict: Decoded JSON data with proper Indic text rendering
    """
    with open(path, 'r', encoding='utf-8') as file:
        data = json.load(file)
        
        # Handle the text fields with proper Unicode handling
        if 'header' in data:
            data['header'] = data['header'].encode('utf-8').decode('utf-8')
        if 'full_text' in data:
            data['full_text'] = data['full_text'].encode('utf-8').decode('utf-8')
            
        return data

In [139]:
file_id_gt_dict = []
for file_gt in path_gt_single_pg:
    file_nm = file_gt.name.split(".")[0]
    gt_json = read_json(file_gt)
    file_id_gt_dict.append({"file_id": file_nm, "ground_truth": (gt_json['header'] + "\n" + gt_json['full_text']).replace("\n", " ")})

In [140]:
gt_df = pd.DataFrame(file_id_gt_dict)

In [145]:
gt_df_ocr = pd.merge(gt_df, ocr_res_df, left_on='file_id', right_on='file_id')

In [146]:
gt_df_ocr

,file_id,ground_truth,ocr_output_L_0,ocr_output_L_1,ocr_output_L_2,ocr_output_L_3
0,Bengali_Baloo_Da_2_9,একটি মাধ্যমে আলফা যা ভেতর সফটওয়্যার রিলিজ হল ...,একটি মাধ্যমে আলফা যা ভেতর\nসফটওয়্যার রিলিজ হল...,একটি মাধ্যমে আলফা যা ভেতর\nসফটওয়্যার রিলিজ হল...,একটি মাধ্যমে আলফা যা ভেতর\nসফটওয়্যার রিলিজ হল...,একটি মাধ্যমে আলফা যা ভেতর\nসফটওয়্যার রিলিজ হল...
1,Bengali_Atma_30,server-এর এমন করে রাখলে সকল গেটওয়ে হলো এমন ধর...,server-এর এমন করে রাখলে সকল\nগেটওয়ে হলো এমন ধ...,server-এর এমন করে রাখলে সকল\nগেটওয়ে হলো এমন ধ...,server-এর এমন করে রাখলে সকল\nগেটওয়ে হলো এমন ধর...,server-এর এমন করে রাখলে সকল\nগেটওয়ে হলো এমন ধ...
2,Bengali_Atma_27,শিকারকে ক্ষতির ধরনের সামনের বিষের কলুব্রিডি ()...,শিকারকে ক্ষতির ধরনের সামনের বিষের\nকলব্রিডি ()...,শিকারকে ক্ষতির ধরনের সামনের বিষের\nকলব্রিডি ()...,শিকারকে ক্ষতির ধরনের সামনের বিষের\nকলুব্রিডি (...,শিকারকে ক্ষতির ধরনের সামনের বিষের\nকলুব্রিডি (...
3,Bengali_Baloo_Da_2_29,ইডি দলের তৃণমূল কংগ্রেস তিনি পার্থ চট্টোপাধ্যা...,ইডি দলের তৃণমূল কংগ্রেস তিনি\nপার্থ চট্টোপাধ্য...,ইডি দলের তৃণমূল কংগ্রেস তিনি\nপার্থ চট্টোপাধ্য...,ইডি দলের তৃণমূল কংগ্রেস তিনি\nপার্থ চট্টোপাধ্য...,ইডি দলের তৃণমূল কংগ্রেস তিনি\nপার্থ চট্টোপাধ্য...
4,Bengali_Atma_20,করেন। ছিল ছিলেন। আগস্ট হলে আইভি রহমান (৭ জুলাই...,করেন। ছিল ছিলেন। আগস্ট হলে\nআইভি রহমান (৭ জুলা...,করেন। ছিল ছিলেন। আগস্ট হলে\nআইভি রহমান (৭ জুলা...,করেন। ছিল ছিলেন। আগস্ট হলে\nআইভি রহমান (৭ জুলা...,করেন। ছিল ছিলেন। আগস্ট হলে\nআইভি রহমান (৭ জুলা...
5,Bengali_Baloo_Da_2_3,দেখিয়েছেন পাঞ্জাব ১৯৬৭ মোহাম্মদ শাস্ত্রীয় মো...,দেখিয়েছেন পাঞ্জাব ১৯৬৭ মোহাম্মদ শাস্ত্রীয়\nম...,দেখিয়েছেন পাঞ্জাব ১৯৬৭ মোহাম্মদ শাস্ত্রীয়\nম...,দেখিয়েছেন পাঞ্জাব ১৯৬৭ মোহাম্মদ শাস্ত্রীয়\nমো...,দেখিয়েছেন পাঞ্জাব ১৯৬৭ মোহাম্মদ শাস্ত্রীয়\nম...
6,Bengali_Noto_Sans_Bengali_11,সালে পরিচালিত একটি যুদ্ধের তুলেছিলেন। গেরিলা ২...,সালে পরিচালিত একটি যুদ্ধের তুলেছিলেন।\nগেরিলা ...,সালে পরিচালিত একটি যুদ্ধের তুলেছিলেন।\nগেরিলা ...,সালে পরিচালিত একটি যুদ্ধের তুলেছিলেন।\nগেরিলা ...,সালে পরিচালিত একটি যুদ্ধের তুলেছিলেন।\nগেরিলা ...
7,Bengali_Atma_1,মাধ্যমে শিকার পছন্দ ঘন্টা খাদ্য গোশত বা মাংস হ...,মাধ্যমে শিকার পছন্দ ঘন্টা খাদ্য\nগোশত বা মাংস ...,মাধ্যমে শিকার পছন্দ ঘন্টা খাদ্য\nগোশত বা মাংস ...,মাধ্যমে শিকার পছন্দ ঘন্টা খাদ্য\nগোশত বা মাংস ...,মাধ্যমে শিকার পছন্দ ঘন্টা খাদ্য\nগোশত বা মাংস ...
8,Bengali_Atma_17,"পৌঁছায়, আর হলেও বেগে অতিআলোকীয় অতিআলোকীয় গত...","পৌঁছায়, আর হলেও বেগে অতিআলোকীয়\nঅতিআলোকীয় গতি ...","পৌঁছায়, আর হলেও বেগে অতিআলোকীয়\nঅতিআলোকীয় গত...","পৌঁছায়, আর হলেও বেগে অতিআলোকীয়\nঅতিআলোকীয় গতি...","পৌঁছায়, আর হলেও বেগে অতিআলোকীয়\nঅতিআলোকীয় গ..."
9,Bengali_Baloo_Da_2_15,তত্ত্বে সবচেয়ে লরেনৎস সময়ের নির্ভর দুটি ঘটনা...,তত্ত্বে সবচেয়ে লরেনৎস সময়ের নির্ভর\nদুটি ঘটনা...,তত্ত্বে সবচেয়ে লরেনৎস সময়ের নির্ভর\nদুটি ঘটন...,তত্ত্বে সবচেয়ে লরেনৎস সময়ের নির্ভর\nদুটি ঘটন...,তত্ত্বে সবচেয়ে লরেনৎস সময়ের নির্ভর\nদুটি ঘটন...
